# 법령 PDF 10개 → LLM 자유 추출 → Neo4j 적재

정해진 스키마(allowed_nodes, allowed_relationships) 없이, PDF 하나를 통째로 LLM에 넣고 노드/관계 타입을 LLM이 알아서 판단하게 합니다.

**주의**
- PDF 하나가 너무 길면(대략 25,000토큰 이상) TPM 한도(30,000)에 걸릴 수 있어, 그 경우에만 자동으로 절반씩 나눠서 2번 호출합니다.
- 서로 다른 법령에 같은 이름의 노드 id(예: '제3조')가 생기는 걸 막기 위해, 최종 노드 id 앞에 PDF 파일명을 붙입니다 (모델이 만든 원래 id는 `original_id` 속성으로 보존).

In [28]:
import os
import re
import json
import time
import glob

from dotenv import load_dotenv
load_dotenv()

PDF_DIR = "data/"          # 법령 PDF 10개가 들어있는 폴더
TARGET_DATABASE = "lawdb"
MODEL_NAME = "gpt-4o-mini"
MAX_INPUT_TOKENS = 25000        # 이 이상이면 절반씩 나눠서 호출

In [29]:
# 필요 패키지: pip install pymupdf openai neo4j python-dotenv tiktoken --break-system-packages
import fitz  # PyMuPDF
import tiktoken
from openai import OpenAI
from neo4j import GraphDatabase

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
driver = GraphDatabase.driver(
    os.getenv('NEO4J_URI'),
    auth=(os.getenv('NEO4J_USERNAME'), os.getenv('NEO4J_PASSWORD'))
)
encoding = tiktoken.encoding_for_model("gpt-4o")

def count_tokens(text: str) -> int:
    return len(encoding.encode(text))

In [30]:
def extract_pdf_text(pdf_path: str) -> str:
    """PDF 전체 텍스트를 페이지 순서대로 이어붙여 반환"""
    doc = fitz.open(pdf_path)
    pages = [page.get_text() for page in doc]
    doc.close()
    return "\n".join(pages)

In [31]:
FREEFORM_SYSTEM_PROMPT = """
당신은 법령 문서를 분석해 지식 그래프를 추출하는 전문가입니다.
정해진 노드/관계 스키마는 없습니다. 문서 내용을 읽고, 가장 자연스럽고 유용하다고
판단되는 노드 타입(label)과 관계 타입(relationship type)을 스스로 정해서 추출하세요.

반드시 아래 JSON 형식으로만 출력하세요. 다른 설명이나 마크다운 코드블록 표시는 절대 넣지 마세요.

{
  "nodes": [
    {"id": "고유식별자", "label": "노드타입", "properties": {"name": "...", "description": "..."}}
  ],
  "relationships": [
    {"source": "노드id", "target": "노드id", "type": "관계타입", "properties": {}}
  ]
}

규칙:
1. id는 문서 내에서 유일해야 합니다 (예: 조문이면 '제3조', '제19조' 등).
2. label과 type은 한국어든 영어든 상관없지만, 일관된 스타일(예: 영어 대문자 스네이크케이스)을 유지하세요.
3. 본문에 명확한 근거가 없는 노드/관계는 절대 만들지 마세요.
4. 조문 본문 내용은 properties.description에 최대한 원문 그대로 담아, 나중에 내용 검색에 쓸 수 있게 하세요.
5. relationships의 source/target은 반드시 nodes 목록에 있는 id와 일치해야 합니다.
"""

def call_llm_freeform(text: str, model=MODEL_NAME):
    response = client.chat.completions.create(
        model=model,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": FREEFORM_SYSTEM_PROMPT},
            {"role": "user", "content": text},
        ],
    )
    raw = response.choices[0].message.content
    return json.loads(raw)

In [32]:
def call_with_backoff(text: str, max_retries=6):
    for attempt in range(max_retries):
        try:
            return call_llm_freeform(text)
        except Exception as e:
            msg = str(e)
            if "402" in msg or "insufficient_quota" in msg.lower():
                print("\U0001F4B3 402 에러: 결제/쿼터 문제. 중단합니다.")
                raise
            if "429" in msg or "rate_limit" in msg.lower():
                wait = min(60, 2 ** attempt)
                print(f"\u23F3 429 감지 (시도 {attempt+1}/{max_retries}). {wait}초 대기...")
                time.sleep(wait)
                continue
            raise
    raise RuntimeError("재시도 한도를 초과했습니다.")


def extract_graph_for_pdf(pdf_path: str) -> dict:
    """PDF 하나를 통째로 LLM에 넣어 노드/관계 추출. 너무 길면 절반씩 나눠서 병합."""
    text = extract_pdf_text(pdf_path)
    tokens = count_tokens(text)
    print(f"  텍스트 길이: 약 {tokens} 토큰")

    if tokens <= MAX_INPUT_TOKENS:
        return call_with_backoff(text)

    print(f"  \u26A0 토큰 한도 초과 → 절반씩 나눠서 2회 호출")
    mid = len(text) // 2
    part1, part2 = text[:mid], text[mid:]

    result1 = call_with_backoff(part1)
    time.sleep(2)
    result2 = call_with_backoff(part2)

    merged = {
        "nodes": result1.get("nodes", []) + result2.get("nodes", []),
        "relationships": result1.get("relationships", []) + result2.get("relationships", []),
    }
    return merged

In [33]:
def sanitize_label(name: str) -> str:
    """Cypher 라벨/관계타입으로 안전하게 쓰도록 특수문자 제거"""
    cleaned = re.sub(r'[^0-9a-zA-Z가-힣_]', '_', name.strip())
    if not cleaned or cleaned[0].isdigit():
        cleaned = f"N_{cleaned}"
    return cleaned


def load_graph_to_neo4j(graph_data: dict, pdf_stem: str, session):
    """자유 추출된 노드/관계를 Neo4j에 적재. id 앞에 파일명을 붙여 문서 간 충돌 방지."""
    node_count, rel_count = 0, 0

    for node in graph_data.get("nodes", []):
        original_id = node.get("id")
        if not original_id:
            continue
        final_id = f"{pdf_stem}::{original_id}"
        label = sanitize_label(node.get("label", "Entity"))
        properties = dict(node.get("properties", {}))
        properties["original_id"] = original_id
        properties["source_pdf"] = pdf_stem

        query = f"MERGE (n:{label} {{id: $id}}) SET n += $properties"
        session.run(query, id=final_id, properties=properties)
        node_count += 1

    for rel in graph_data.get("relationships", []):
        source = rel.get("source")
        target = rel.get("target")
        rel_type = rel.get("type")
        if not (source and target and rel_type):
            continue

        final_source = f"{pdf_stem}::{source}"
        final_target = f"{pdf_stem}::{target}"
        safe_type = sanitize_label(rel_type)
        properties = rel.get("properties", {})

        query = f"""
        MATCH (a {{id: $source}}), (b {{id: $target}})
        MERGE (a)-[r:{safe_type}]->(b)
        SET r += $properties
        """
        result = session.run(query, source=final_source, target=final_target, properties=properties)
        rel_count += 1

    return node_count, rel_count

In [34]:
pdf_paths = sorted(glob.glob(os.path.join(PDF_DIR, "*.pdf")))
print(f"발견된 PDF: {len(pdf_paths)}개")
for p in pdf_paths:
    print(f"  - {p}")

발견된 PDF: 10개
  - data\공무원보수규정(대통령령)(제36501호)(20260707).pdf
  - data\군인 징계령 시행규칙(국방부령)(제01203호)(20260128).pdf
  - data\군인보수법(법률)(제20802호)(20260319).pdf
  - data\군인사법(법률)(제21319호)(20260203).pdf
  - data\군인의 지위 및 복무에 관한 기본법 시행령(대통령령)(제35925호)(20260108).pdf
  - data\군인의 지위 및 복무에 관한 기본법(법률)(제20641호)(20260108).pdf
  - data\군형법(법률)(제18465호)(20220701).pdf
  - data\병역법 시행령(대통령령)(제35948호)(20260102).pdf
  - data\병역법(법률)(제21769호)(20260609).pdf
  - data\부대관리훈령(국방부훈령)(제3148호)(20260319).pdf


In [35]:
with driver.session(database=TARGET_DATABASE) as session:
    for pdf_path in pdf_paths:
        pdf_stem = os.path.splitext(os.path.basename(pdf_path))[0]
        print(f"\n\U0001F4C4 처리 중: {pdf_stem}")

        try:
            graph_data = extract_graph_for_pdf(pdf_path)
        except Exception as e:
            print(f"  \u274C 추출 실패: {e}")
            continue

        node_count, rel_count = load_graph_to_neo4j(graph_data, pdf_stem, session)
        print(f"  \u2705 적재 완료: 노드 {node_count}개, 관계 {rel_count}개")

        time.sleep(2)

print("\n\U0001F389 전체 PDF 처리 완료!")


📄 처리 중: 공무원보수규정(대통령령)(제36501호)(20260707)
  텍스트 길이: 약 32312 토큰
  ⚠ 토큰 한도 초과 → 절반씩 나눠서 2회 호출
  ✅ 적재 완료: 노드 22개, 관계 20개

📄 처리 중: 군인 징계령 시행규칙(국방부령)(제01203호)(20260128)
  텍스트 길이: 약 3968 토큰
  ✅ 적재 완료: 노드 10개, 관계 9개

📄 처리 중: 군인보수법(법률)(제20802호)(20260319)
  텍스트 길이: 약 3016 토큰
  ✅ 적재 완료: 노드 27개, 관계 26개

📄 처리 중: 군인사법(법률)(제21319호)(20260203)
  텍스트 길이: 약 42655 토큰
  ⚠ 토큰 한도 초과 → 절반씩 나눠서 2회 호출
  ✅ 적재 완료: 노드 48개, 관계 40개

📄 처리 중: 군인의 지위 및 복무에 관한 기본법 시행령(대통령령)(제35925호)(20260108)
  텍스트 길이: 약 16937 토큰
  ✅ 적재 완료: 노드 28개, 관계 25개

📄 처리 중: 군인의 지위 및 복무에 관한 기본법(법률)(제20641호)(20260108)
  텍스트 길이: 약 11061 토큰
  ✅ 적재 완료: 노드 53개, 관계 52개

📄 처리 중: 군형법(법률)(제18465호)(20220701)
  텍스트 길이: 약 16153 토큰
  ✅ 적재 완료: 노드 100개, 관계 51개

📄 처리 중: 병역법 시행령(대통령령)(제35948호)(20260102)
  텍스트 길이: 약 134387 토큰
  ⚠ 토큰 한도 초과 → 절반씩 나눠서 2회 호출
  ✅ 적재 완료: 노드 174개, 관계 171개

📄 처리 중: 병역법(법률)(제21769호)(20260609)
  텍스트 길이: 약 71431 토큰
  ⚠ 토큰 한도 초과 → 절반씩 나눠서 2회 호출
  ✅ 적재 완료: 노드 52개, 관계 44개

📄 처리 중: 부대관리훈령(국방부훈령)(제3148호)(20260319)
  텍스트 길이: 약 77552 토큰
  ⚠ 토큰 한도 초

In [37]:
###마지막 pdf만 out of token으로 추출실패 해서 마지막만 다시 돌리기.
with driver.session(database=TARGET_DATABASE) as session:
        pdf_path = "data/부대관리훈령(국방부훈령)(제3148호)(20260319).pdf"
        pdf_stem = os.path.splitext(os.path.basename(pdf_path))[0]
        print(f"\n\U0001F4C4 처리 중: {pdf_stem}")

        try:
            graph_data = extract_graph_for_pdf(pdf_path)
        except Exception as e:
            print(f"  \u274C 추출 실패: {e}")

        node_count, rel_count = load_graph_to_neo4j(graph_data, pdf_stem, session)
        print(f"  \u2705 적재 완료: 노드 {node_count}개, 관계 {rel_count}개")

        time.sleep(2)

print("\n\U0001F389 전체 PDF 처리 완료!")


📄 처리 중: 부대관리훈령(국방부훈령)(제3148호)(20260319)
  텍스트 길이: 약 77552 토큰
  ⚠ 토큰 한도 초과 → 절반씩 나눠서 2회 호출
  ❌ 추출 실패: Unterminated string starting at: line 227 column 93 (char 34455)
  ✅ 적재 완료: 노드 52개, 관계 44개

🎉 전체 PDF 처리 완료!


In [38]:
# 결과 확인: 어떤 라벨/관계타입이 자유롭게 생성됐는지 확인
with driver.session(database=TARGET_DATABASE) as session:
    print("생성된 노드 라벨:")
    result = session.run("CALL db.labels()")
    for r in result:
        print(f"  - {r['label']}")

    print("\n생성된 관계 타입:")
    result = session.run("CALL db.relationshipTypes()")
    for r in result:
        print(f"  - {r['relationshipType']}")

생성된 노드 라벨:
  - ARTICLE
  - REGULATION
  - SUPPLEMENTARY_PROVISION
  - PROVISION
  - SUPPLEMENTARY
  - SUPPLEMENTARY_PROVISIONS

생성된 관계 타입:
  - NEXT
  - RELATED_TO
  - FOLLOWS
  - REGULATES
  - EXCEPTION
  - CONDITIONS
  - PROVIDES
  - SUPPORTS
  - INFORMS
  - PENALTY


In [39]:
import os
from openai import OpenAI
from neo4j import GraphDatabase

TARGET_DATABASE = "lawdb"
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
driver = GraphDatabase.driver(
    os.getenv('NEO4J_URI'),
    auth=(os.getenv('NEO4J_USERNAME'), os.getenv('NEO4J_PASSWORD'))
)

def embed_all_articles(batch_size=100):
    with driver.session(database=TARGET_DATABASE) as session:
        result = session.run("""
            MATCH (a:ARTICLE)
            WHERE a.description IS NOT NULL AND a.embedding IS NULL
            RETURN a.id AS id, a.description AS description
        """)
        rows = [dict(r) for r in result]

    print(f"임베딩 대상: {len(rows)}개")

    for i in range(0, len(rows), batch_size):
        batch = rows[i:i+batch_size]
        texts = [r["description"] for r in batch]

        response = client.embeddings.create(
            model="text-embedding-3-small",
            input=texts
        )
        embeddings = [d.embedding for d in response.data]

        with driver.session(database=TARGET_DATABASE) as session:
            for row, emb in zip(batch, embeddings):
                session.run("""
                    MATCH (a:ARTICLE {id: $id})
                    CALL db.create.setNodeVectorProperty(a, 'embedding', $embedding)
                """, id=row["id"], embedding=emb)

        print(f"  {i+len(batch)}/{len(rows)} 완료")

embed_all_articles()

# 벡터 인덱스 생성
with driver.session(database=TARGET_DATABASE) as session:
    session.run("""
        CREATE VECTOR INDEX article_vector_index IF NOT EXISTS
        FOR (a:ARTICLE) ON (a.embedding)
        OPTIONS {indexConfig: {
          `vector.dimensions`: 1536,
          `vector.similarity_function`: 'cosine'
        }}
    """)
print("✅ 벡터 인덱스 생성 완료")

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `embedding` does not exist in database `lawdb`. Verify that the spelling is correct.', position=<SummaryInputPosition line=3, column=51, offset=81>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 81, 'line': 3, 'column': 51}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n            MATCH (a:ARTICLE)\n            WHERE a.description IS NOT NULL AND a.embedding IS NULL\n            RETURN a.id AS id, a.description AS description\n        '


임베딩 대상: 549개
  100/549 완료
  200/549 완료
  300/549 완료
  400/549 완료
  500/549 완료
  549/549 완료
✅ 벡터 인덱스 생성 완료


In [40]:
import re

SCHEMA_DESCRIPTION = """
노드 라벨: ARTICLE (법령 조문)
속성:
  - id: string, 형식 "법령명(대통령령)(제N호)(날짜)::제N조" (예: "공무원보수규정(대통령령)(제36501호)(20260707)::제31조")
  - name: string, 조문 제목 (예: "수당의 지급")
  - description: string, 조문 본문 내용
  - original_id: string, 조번호만 (예: "제31조")
  - source_pdf: string, 원본 법령명

벡터 인덱스: article_vector_index (embedding 속성 기준, 의미 검색용)
"""

READ_ONLY_ALLOWED = {"MATCH", "OPTIONAL", "WHERE", "RETURN", "WITH", "ORDER", "LIMIT", "SKIP", "CALL", "YIELD", "UNWIND", "AS"}
FORBIDDEN_KEYWORDS = {"CREATE", "DELETE", "SET", "MERGE", "REMOVE", "DROP", "DETACH"}

def is_safe_cypher(query: str) -> bool:
    upper = query.upper()
    return not any(re.search(rf"\b{kw}\b", upper) for kw in FORBIDDEN_KEYWORDS)


def generate_cypher(user_query: str, model="gpt-4o") -> str:
    system_prompt = f"""
당신은 Neo4j Cypher 쿼리를 작성하는 전문가입니다.
아래 스키마만을 근거로, 사용자 질문에 답하기 위한 Cypher 쿼리를 작성하세요.

{SCHEMA_DESCRIPTION}

규칙:
1. 오직 읽기 쿼리만 작성하세요 (MATCH, RETURN 등). CREATE/DELETE/SET/MERGE는 절대 금지.
2. 조번호나 법령명이 질문에 명시되어 있으면 CONTAINS로 텍스트 매칭하세요.
3. 명시된 조번호/법령명이 없고 주제/내용 기반 질문이면, 아래처럼 벡터 인덱스를 사용하세요:
   CALL db.index.vector.queryNodes('article_vector_index', $top_k, $query_embedding)
   YIELD node, score
   RETURN node.id AS id, node.name AS name, node.description AS description, score
   ORDER BY score DESC
4. 쿼리만 출력하세요. 설명, 마크다운 코드블록 표시(```) 없이 순수 Cypher 텍스트만 출력하세요.
"""
    response = client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query},
        ],
    )
    cypher = response.choices[0].message.content.strip()
    cypher = re.sub(r'^```(cypher)?\s*|\s*```$', '', cypher, flags=re.MULTILINE).strip()
    return cypher


def vector_fallback(user_query: str, top_k=3):
    """text-to-Cypher 실패 시 순수 벡터 검색으로 폴백"""
    response = client.embeddings.create(model="text-embedding-3-small", input=[user_query])
    query_embedding = response.data[0].embedding

    cypher = """
    CALL db.index.vector.queryNodes('article_vector_index', $top_k, $query_embedding)
    YIELD node, score
    RETURN node.id AS id, node.name AS name, node.description AS description, score
    ORDER BY score DESC
    """
    with driver.session(database=TARGET_DATABASE) as session:
        results = session.run(cypher, top_k=top_k, query_embedding=query_embedding)
        return [dict(r) for r in results]


def search_with_llm_cypher(user_query: str, top_k=3):
    cypher = generate_cypher(user_query)
    print(f"  생성된 Cypher:\n{cypher}\n")

    if not is_safe_cypher(cypher):
        print("  ⚠ 안전하지 않은 쿼리 감지 → 벡터 검색으로 폴백")
        return vector_fallback(user_query, top_k)

    params = {"top_k": top_k}
    if "query_embedding" in cypher:
        response = client.embeddings.create(model="text-embedding-3-small", input=[user_query])
        params["query_embedding"] = response.data[0].embedding

    try:
        with driver.session(database=TARGET_DATABASE) as session:
            results = session.run(cypher, **params)
            return [dict(r) for r in results]
    except Exception as e:
        print(f"  ⚠ 쿼리 실행 실패 ({e}) → 벡터 검색으로 폴백")
        return vector_fallback(user_query, top_k)

In [57]:
def build_context(search_data):
    blocks = []
    for r in search_data:
        blocks.append(f"[{r.get('id', '')}] {r.get('name', '')}\n{r.get('description', '')}")
    return "\n\n---\n\n".join(blocks)


def generate_rag_answer(user_query, context_text, model="gpt-4o"):
    system_prompt = (
        "법률 내용을 알기 쉽게 설명해주는 전문가입니다. "
        "참조 문서 내용을 바탕으로 자연스럽게 설명하고, 조문 나열은 피하세요. "
        "근거 조문은 답변 끝에 짧게만 덧붙이세요."
    )
    response = client.chat.completions.create(
        model=model, temperature=0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"[참조 문서]\n{context_text}\n\n[질문]\n{user_query}"},
        ],
    )
    return response.choices[0].message.content


# ── 테스트 ──
sample_question = "위병초병은 뭘 지니고 있어야해됨?"
print(f"🔮 질문: {sample_question}\n")

search_data = search_with_llm_cypher(sample_question, top_k=3)
print(f"검색 결과 {len(search_data)}개")

if search_data:
    context = build_context(search_data)
    answer = generate_rag_answer(sample_question, context)
    print(f"\n✅ 답변:\n{answer}")
else:
    print("검색 결과 없음")

🔮 질문: 위병초병은 뭘 지니고 있어야해됨?

  생성된 Cypher:
CALL db.index.vector.queryNodes('article_vector_index', 5, $query_embedding)
YIELD node, score
RETURN node.id AS id, node.name AS name, node.description AS description, score
ORDER BY score DESC



Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.vector.queryNodes('article_vector_index', 5, $query_embedding)\nYIELD node, score\nRETURN node.id AS id, node.name AS name, node.description AS description, score\nORDER BY score DESC"


검색 결과 5개

✅ 답변:
위병초병은 임무 수행을 위해 소총이나 도검과 같은 무기를 휴대할 수 있습니다. 이 무기는 초병이 자신의 임무를 수행하는 데 필요한 최소한의 범위에서 사용할 수 있도록 규정되어 있습니다. 즉, 초병은 자신의 임무를 수행하기 위해 필요한 경우에 한해 이러한 무기를 사용할 수 있는 권한이 주어집니다. 

(근거: 군인의 지위 및 복무에 관한 기본법 제48조)
